# VisionGuard+ — Training Notebook (Google Colab)
**SEC Satria Data 2026 | Sub-tema Kesehatan**

Dataset: Ocular Disease Recognition (ODIR-5K) via KaggleHub

> **Sebelum mulai:** Pastikan GPU aktif → Runtime → Change runtime type → **T4 GPU** → Save

Urutan: jalankan Cell 0 dulu (setup), lalu Cell 1, 2, dst.

## Cell 0 — Setup: Kaggle API + Install Library
Jalankan sekali saja di awal sesi. Upload file `kaggle.json` dari:
**kaggle.com → foto profil → Settings → API → Create New Token**

In [ ]:
# Upload kaggle.json
from google.colab import files
uploaded = files.upload()  # pilih kaggle.json

import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
os.rename('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle API siap.')

# Install library yang dibutuhkan
!pip install kagglehub timm grad-cam -q
print('Library terinstall.')

## Cell 1 — Download Dataset ODIR-5K + Eksplorasi Struktur
Download pertama kali ~5-10 menit (2.5 GB). Sesi berikutnya dari cache, lebih cepat.

In [ ]:
import os
import kagglehub

# Download full dataset (CSV + 6392 gambar retina)
DATA_ROOT = kagglehub.dataset_download('andrewmvd/ocular-disease-recognition-odir5k')
print('Dataset tersimpan di:', DATA_ROOT)

# Verifikasi struktur folder
for dirpath, dirnames, filenames in os.walk(DATA_ROOT):
    level = dirpath.replace(DATA_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(dirpath)}/')
    if level < 2:
        for f in filenames[:5]:
            print(f'{indent}  {f}')
        if len(filenames) > 5:
            print(f'{indent}  ... ({len(filenames)} files total)')

## Cell 2 — Load CSV Label, Cek Kolom

In [ ]:
import pandas as pd
import glob

csv_path = glob.glob(f'{DATA_ROOT}/**/*.csv', recursive=True)[0]
print('CSV ditemukan di:', csv_path)

df = pd.read_csv(csv_path)
print('Kolom:', df.columns.tolist())
df.head()

## Cell 3 — Mapping ke 4 Penyakit Target VisionGuard+
Kolom label asli ODIR-5K (one-hot): N, D, G, C, A, H, M, O

N=Normal, **D=DR**, **G=Glaukoma**, **C=Katarak**, **A=AMD**, H=Hipertensi, M=Miopia, O=Lainnya

> Catatan: kolom `filepath` di CSV berisi path dari environment pembuat dataset asli — kita ambil nama filenya saja dan cocokkan ke folder `preprocessed_images/`

In [ ]:
TARGET_COLS = ['D', 'G', 'C', 'A']
IMAGE_DIR = os.path.join(DATA_ROOT, 'preprocessed_images')

df['image_filename'] = df['filepath'].apply(os.path.basename)
df['full_image_path'] = df['image_filename'].apply(lambda x: os.path.join(IMAGE_DIR, x))

# Filter hanya baris yang gambarnya benar-benar ada
df_clean = df[df['full_image_path'].apply(os.path.exists)].reset_index(drop=True)
df_clean = df_clean[['full_image_path'] + TARGET_COLS].dropna().reset_index(drop=True)

print('Total baris awal:', len(df))
print('Total baris setelah filter:', len(df_clean))
df_clean.head()

## Cell 3b — Statistik Dataset ODIR-5K
Eksplorasi distribusi penyakit, demografi pasien, dan keseimbangan kelas. Penting untuk bagian **Metodologi** di esai SEC.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print('=' * 60)
print('  STATISTIK DATASET ODIR-5K — VisionGuard+')
print('=' * 60)

print(f'\nTotal pasien (baris CSV) : {len(df):,}')
print(f'Total gambar valid       : {len(df_clean):,}')

age_col = df['Patient Age'].dropna()
print(f'\nUsia pasien:')
print(f'  Rata-rata  : {age_col.mean():.1f} tahun')
print(f'  Median     : {age_col.median():.0f} tahun')
print(f'  Std Dev    : {age_col.std():.1f} tahun')
print(f'  Min – Max  : {int(age_col.min())} – {int(age_col.max())} tahun')

gender = df['Patient Sex'].value_counts()
print(f'\nJenis kelamin:')
for g, c in gender.items():
    print(f'  {g:8}: {c:,} ({c/len(df)*100:.1f}%)')

disease_map = {'D': 'DR (Retinopati Diabetik)', 'G': 'Glaukoma',
               'C': 'Katarak', 'A': 'AMD'}
print(f'\nPrevalensi penyakit target (dari {len(df_clean):,} gambar):')
for col, name in disease_map.items():
    n = int(df_clean[col].sum())
    pct = n / len(df_clean) * 100
    bar = '█' * int(pct / 2)
    print(f'  {name:30}: {n:4,} ({pct:5.1f}%)  {bar}')

n_normal = int((df_clean[TARGET_COLS].sum(axis=1) == 0).sum())
print(f'  {"Normal (tidak ada dari 4 penyakit)":30}: {n_normal:4,} ({n_normal/len(df_clean)*100:5.1f}%)')
n_multi = int((df_clean[TARGET_COLS].sum(axis=1) > 1).sum())
print(f'  {"Multi-label (>1 penyakit)":30}: {n_multi:4,} ({n_multi/len(df_clean)*100:5.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('VisionGuard+ — Statistik Dataset ODIR-5K', fontsize=14, fontweight='bold')

ax1 = axes[0]
labels_dis = ['DR', 'Glaukoma', 'Katarak', 'AMD']
counts_dis = [int(df_clean[c].sum()) for c in TARGET_COLS]
colors_dis = ['#E74C3C', '#27AE60', '#3498DB', '#9B59B6']
bars = ax1.bar(labels_dis, counts_dis, color=colors_dis, alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, c in zip(bars, counts_dis):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
             f'{c:,}\n({c/len(df_clean)*100:.1f}%)', ha='center', fontsize=10, fontweight='bold')
ax1.set_title('Prevalensi per Penyakit Target', fontsize=12)
ax1.set_ylabel('Jumlah Kasus')
ax1.set_ylim(0, max(counts_dis) * 1.2)
ax1.grid(True, alpha=0.3, axis='y')

ax2 = axes[1]
ax2.hist(age_col, bins=20, color='#3498DB', alpha=0.75, edgecolor='white', linewidth=0.8)
ax2.axvline(age_col.mean(), color='#E74C3C', linestyle='--', linewidth=2,
            label=f'Rata-rata: {age_col.mean():.0f} tahun')
ax2.axvline(age_col.median(), color='#F39C12', linestyle='--', linewidth=2,
            label=f'Median: {age_col.median():.0f} tahun')
ax2.set_title('Distribusi Usia Pasien', fontsize=12)
ax2.set_xlabel('Usia (tahun)')
ax2.set_ylabel('Jumlah Pasien')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

ax3 = axes[2]
gender_counts = df['Patient Sex'].value_counts()
ax3.pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
        colors=['#3498DB', '#E91E63'], startangle=90,
        textprops={'fontsize': 11}, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax3.set_title('Distribusi Jenis Kelamin', fontsize=12)

plt.tight_layout()
plt.savefig('dataset_statistics_visionguard.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nStatistik dataset disimpan: dataset_statistics_visionguard.png')

## Cell 4 — Train/Val Split + Dataset Class

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image
import torchvision.transforms as T

train_df, val_df = train_test_split(df_clean, test_size=0.15, random_state=42)
print('Train:', len(train_df), '| Val:', len(val_df))

IMG_SIZE = 380  # resolusi native EfficientNet-B4

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class ODIRDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['full_image_path']).convert('RGB')
        img = self.transform(img)
        labels = torch.tensor(row[TARGET_COLS].values.astype('float32'))
        return img, labels

train_ds = ODIRDataset(train_df, train_tf)
val_ds   = ODIRDataset(val_df,   val_tf)

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print('DataLoader siap.')

## Cell 5 — Model: EfficientNet-B4 Multi-Head + Dropout
> Pastikan GPU aktif: Runtime → Change runtime type → T4 GPU

In [ ]:
import timm
import torch.nn as nn

class VisionGuardModel(nn.Module):
    def __init__(self, num_diseases=4, dropout=0.3):
        super().__init__()
        # EfficientNet-B4 sudah punya SE-block built-in di tiap MBConv block
        self.backbone = timm.create_model('efficientnet_b4', pretrained=True, num_classes=0)
        feat_dim = self.backbone.num_features
        self.dropout = nn.Dropout(p=dropout)  # regularisasi untuk kurangi overfitting

        self.heads = nn.ModuleDict({
            'DR':       nn.Linear(feat_dim, 1),
            'Glaukoma': nn.Linear(feat_dim, 1),
            'Katarak':  nn.Linear(feat_dim, 1),
            'AMD':      nn.Linear(feat_dim, 1),
        })

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.dropout(feat)
        return {name: head(feat).squeeze(-1) for name, head in self.heads.items()}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = VisionGuardModel().to(device)
print('Model siap. Parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad):,)

## Cell 6 — Loss Function (Weighted Multi-Task) + Optimizer + Scheduler

In [ ]:
HEAD_NAMES   = ['DR', 'Glaukoma', 'Katarak', 'AMD']
HEAD_WEIGHTS = {'DR': 0.4, 'Glaukoma': 0.2, 'Katarak': 0.2, 'AMD': 0.2}

bce_loss = nn.BCEWithLogitsLoss()

def compute_loss(outputs, labels):
    label_map = {'DR': labels[:, 0], 'Glaukoma': labels[:, 1],
                 'Katarak': labels[:, 2], 'AMD': labels[:, 3]}
    total = 0.0
    for name in HEAD_NAMES:
        total += HEAD_WEIGHTS[name] * bce_loss(outputs[name], label_map[name])
    return total

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)
print('Optimizer dan scheduler siap.')

## Cell 7 — Training Loop (Early Stopping + Save Best Model)
Estimasi waktu: **~1–2 jam** di GPU T4. Jangan tutup tab Colab.

Perbaikan overfitting: Dropout(0.3) + weight_decay=1e-4 + ReduceLROnPlateau + Early Stopping (patience=5)

In [ ]:
EPOCHS   = 20
PATIENCE = 5
os.makedirs('checkpoints', exist_ok=True)

best_val_loss     = float('inf')
epochs_no_improve = 0
BEST_MODEL_PATH   = 'visionguard_best.pt'
history = {'train_loss': [], 'val_loss': [], 'epoch': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(imgs)
            loss    = compute_loss(outputs, labels)

        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                outputs = model(imgs)
                loss    = compute_loss(outputs, labels)
            val_loss += loss.item()
    val_loss /= len(val_loader)

    scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['epoch'].append(epoch)

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss     = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        status = '  [BEST - disimpan]'
    else:
        epochs_no_improve += 1
        status = f'  [no improve {epochs_no_improve}/{PATIENCE}]'

    print(f'Epoch {epoch}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}{status}')
    torch.save(model.state_dict(), f'checkpoints/epoch_{epoch}.pt')

    if epochs_no_improve >= PATIENCE:
        print(f'\nEarly stopping dipicu di epoch {epoch}. Best val loss: {best_val_loss:.4f}')
        break

# Load model terbaik
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
torch.save(model.state_dict(), 'visionguard_model_final.pt')
print(f'\nTraining selesai. Best val loss: {best_val_loss:.4f}')
print('Model terbaik: visionguard_model_final.pt')

## Cell 7b — Training Curve (Loss per Epoch)
Jalankan langsung setelah Cell 7 selesai.

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = history['epoch']
best_epoch = history['val_loss'].index(min(history['val_loss'])) + 1
best_loss  = min(history['val_loss'])

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(epochs_ran, history['train_loss'], 'b-o', linewidth=2, markersize=5, label='Train Loss')
ax.plot(epochs_ran, history['val_loss'],   'r-o', linewidth=2, markersize=5, label='Validation Loss')
ax.axvline(best_epoch, color='#27AE60', linestyle='--', linewidth=2, alpha=0.8,
           label=f'Best epoch {best_epoch} (val loss = {best_loss:.4f})')
ax.fill_between(epochs_ran, history['train_loss'], history['val_loss'],
                alpha=0.08, color='purple', label='Overfitting gap')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title(
    f'VisionGuard+ — Training & Validation Loss\n'
    f'EfficientNet-B4 + Dropout(0.3) | ODIR-5K | Best epoch: {best_epoch}',
    fontsize=13
)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(1, max(epochs_ran))

plt.tight_layout()
plt.savefig('training_curve_visionguard.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Training curve disimpan: training_curve_visionguard.png')
print(f'Best epoch    : {best_epoch}')
print(f'Best val loss : {best_loss:.4f}')

## Cell 8 — Evaluasi Lengkap: AUC + Sensitivitas + Spesifisitas + F1
Jalankan HANYA setelah Cell 7 selesai. Hasil ini yang dipakai di esai SEC.

In [ ]:
from sklearn.metrics import roc_auc_score, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

model.eval()
all_labels = {name: [] for name in HEAD_NAMES}
all_probs  = {name: [] for name in HEAD_NAMES}

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        lmap = {'DR': labels[:, 0], 'Glaukoma': labels[:, 1],
                'Katarak': labels[:, 2], 'AMD': labels[:, 3]}
        for name in HEAD_NAMES:
            all_probs[name].extend(torch.sigmoid(outputs[name]).cpu().numpy())
            all_labels[name].extend(lmap[name].numpy())

THRESHOLD     = 0.5
disease_colors = {'DR': '#E74C3C', 'Glaukoma': '#27AE60', 'Katarak': '#3498DB', 'AMD': '#9B59B6'}

print('=' * 65)
print('  VisionGuard+ — EVALUASI LENGKAP (Validation Set: 959 gambar)')
print('=' * 65)

metrics = {}
for name in HEAD_NAMES:
    y_true = np.array(all_labels[name]).astype(int)
    y_prob = np.array(all_probs[name])
    y_pred = (y_prob >= THRESHOLD).astype(int)

    auc  = roc_auc_score(y_true, y_prob)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = 2 * prec * sens / (prec + sens) if (prec + sens) > 0 else 0.0
    kat  = 'Excellent' if auc >= 0.9 else 'Good' if auc >= 0.8 else 'Fair'

    metrics[name] = {'auc': auc, 'sens': sens, 'spec': spec,
                     'prec': prec, 'f1': f1, 'kat': kat,
                     'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn}

    print(f'\n  [{name}]  AUC = {auc:.4f}  [{kat}]')
    print(f'    Sensitivitas (Recall) : {sens:.3f}')
    print(f'    Spesifisitas          : {spec:.3f}')
    print(f'    Presisi               : {prec:.3f}')
    print(f'    F1-Score              : {f1:.3f}')
    print(f'    Confusion Matrix      : TP={tp} | FP={fp} | TN={tn} | FN={fn}')

avg_auc  = np.mean([m['auc']  for m in metrics.values()])
avg_sens = np.mean([m['sens'] for m in metrics.values()])
avg_spec = np.mean([m['spec'] for m in metrics.values()])
avg_f1   = np.mean([m['f1']   for m in metrics.values()])

print(f'\n{"=" * 65}')
print(f'  RATA-RATA  AUC={avg_auc:.4f} | Sens={avg_sens:.3f} | Spec={avg_spec:.3f} | F1={avg_f1:.3f}')
print(f'{"=" * 65}')

# Tabel metrik visualisasi
fig, ax = plt.subplots(figsize=(13, 4))
ax.axis('off')
rows = []
for name in HEAD_NAMES:
    m = metrics[name]
    rows.append([name, f"{m['auc']:.4f}", f"{m['sens']:.3f}", f"{m['spec']:.3f}",
                 f"{m['prec']:.3f}", f"{m['f1']:.3f}", m['kat']])
rows.append(['RATA-RATA', f'{avg_auc:.4f}', f'{avg_sens:.3f}', f'{avg_spec:.3f}', '—', f'{avg_f1:.3f}', ''])

tbl = ax.table(
    cellText=rows,
    colLabels=['Penyakit', 'AUC-ROC', 'Sensitivitas', 'Spesifisitas', 'Presisi', 'F1-Score', 'Kategori'],
    loc='center', cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.1, 2.8)
for j in range(7):
    tbl[0, j].set_facecolor('#2C3E50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')
for i, name in enumerate(HEAD_NAMES):
    tbl[i+1, 0].set_text_props(color=disease_colors[name], fontweight='bold')
for j in range(7):
    tbl[len(HEAD_NAMES)+1, j].set_facecolor('#BDC3C7')
    tbl[len(HEAD_NAMES)+1, j].set_text_props(fontweight='bold')

ax.set_title('VisionGuard+ — Ringkasan Metrik Evaluasi (Validation Set ODIR-5K)',
             fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('metrics_table_visionguard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tabel metrik disimpan: metrics_table_visionguard.png')

## Cell 9 — Load Model Terlatih (Sesi Baru)
Jalankan cell ini **hanya** jika memulai sesi baru di Colab (runtime di-restart).
Pastikan Cell 0–4 sudah jalan dulu. Skip jika masih dalam sesi yang sama setelah Cell 7.

In [ ]:
import timm
import torch
import torch.nn as nn

class VisionGuardModel(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b4', pretrained=False, num_classes=0)
        feat_dim = self.backbone.num_features
        self.dropout = nn.Dropout(p=dropout)  # harus sama dengan Cell 5
        self.heads = nn.ModuleDict({
            'DR':       nn.Linear(feat_dim, 1),
            'Glaukoma': nn.Linear(feat_dim, 1),
            'Katarak':  nn.Linear(feat_dim, 1),
            'AMD':      nn.Linear(feat_dim, 1),
        })
    def forward(self, x):
        feat = self.backbone(x)
        feat = self.dropout(feat)
        return {name: head(feat).squeeze(-1) for name, head in self.heads.items()}

HEAD_NAMES = ['DR', 'Glaukoma', 'Katarak', 'AMD']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MODEL_PATH = 'visionguard_model_final.pt'
model = VisionGuardModel().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f'Model dimuat dari: {MODEL_PATH}')
print(f'Device: {device}')
print('Siap untuk Grad-CAM, ROC Curve, dan Risk Stratification.')

## Cell 10 — Grad-CAM: Explainability Heatmap Retina
Visualisasi area retina yang paling berpengaruh terhadap prediksi CNN. Semua gambar dari dataset ODIR-5K asli.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import matplotlib.pyplot as plt
import numpy as np
import torchvision.transforms as T
from PIL import Image

class GradCAMWrapper(nn.Module):
    def __init__(self, base_model, disease='DR'):
        super().__init__()
        self.base    = base_model
        self.disease = disease
    def forward(self, x):
        return self.base(x)[self.disease].unsqueeze(1)

cam_model     = GradCAMWrapper(model, disease='DR')
target_layers = [model.backbone.conv_head]

samples_pos = val_df[val_df['D'] == 1].sample(3, random_state=42)
samples_neg = val_df[val_df['D'] == 0].sample(3, random_state=42)
samples     = pd.concat([samples_pos, samples_neg]).reset_index(drop=True)
true_labels = ['DR+'] * 3 + ['DR-'] * 3

transform_gcam = T.Compose([
    T.Resize((380, 380)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

fig, axes = plt.subplots(2, 6, figsize=(24, 8))
fig.suptitle(
    'VisionGuard+ Stage 1 — Grad-CAM Explainability\n'
    'Baris atas: foto retina asli ODIR-5K  |  Baris bawah: heatmap area kritis\n'
    '(Merah = sangat berpengaruh untuk prediksi DR, Biru = tidak berpengaruh)',
    fontsize=12, fontweight='bold', y=1.02
)

with GradCAM(model=cam_model, target_layers=target_layers) as cam:
    for i, (_, row) in enumerate(samples.iterrows()):
        img_pil     = Image.open(row['full_image_path']).convert('RGB')
        img_resized = img_pil.resize((380, 380))
        img_tensor  = transform_gcam(img_pil).unsqueeze(0).to(device)
        heatmap     = cam(input_tensor=img_tensor, targets=None)[0]
        img_np      = np.array(img_resized) / 255.0
        visualization = show_cam_on_image(img_np, heatmap, use_rgb=True)
        with torch.no_grad():
            prob_dr = torch.sigmoid(model(img_tensor)['DR']).item()
        status = 'RUJUK' if prob_dr >= 0.5 else 'AMAN'
        color  = '#E74C3C' if status == 'RUJUK' else '#27AE60'
        axes[0][i].imshow(img_resized)
        axes[0][i].set_title(f'Label: {true_labels[i]}', fontsize=10, fontweight='bold')
        axes[0][i].axis('off')
        axes[1][i].imshow(visualization)
        axes[1][i].set_title(f'Prob DR: {prob_dr:.1%}  →  {status}',
                              fontsize=10, color=color, fontweight='bold')
        axes[1][i].axis('off')

plt.tight_layout()
plt.savefig('gradcam_visionguard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grad-CAM disimpan: gradcam_visionguard.png')
print('(Gunakan gambar ini di esai SEC sebagai bukti explainability Stage 1)')

## Cell 11 — ROC Curve 4 Penyakit
Kurva ROC untuk keempat penyakit target VisionGuard+. Semua prediksi dari model terlatih pada validation set asli.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import numpy as np

model.eval()
all_labels_roc = {name: [] for name in HEAD_NAMES}
all_probs_roc  = {name: [] for name in HEAD_NAMES}

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs    = imgs.to(device)
        outputs = model(imgs)
        lmap    = {'DR': labels[:, 0], 'Glaukoma': labels[:, 1],
                   'Katarak': labels[:, 2], 'AMD': labels[:, 3]}
        for name in HEAD_NAMES:
            all_probs_roc[name].extend(torch.sigmoid(outputs[name]).cpu().numpy())
            all_labels_roc[name].extend(lmap[name].numpy())

disease_colors = {'DR': '#E74C3C', 'Glaukoma': '#27AE60', 'Katarak': '#3498DB', 'AMD': '#9B59B6'}
fig, (ax_roc, ax_tbl) = plt.subplots(1, 2, figsize=(16, 7),
                                      gridspec_kw={'width_ratios': [3, 1]})
ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1.5, label='Random (AUC = 0.50)')

aucs = {}
for name in HEAD_NAMES:
    fpr, tpr, _ = roc_curve(all_labels_roc[name], all_probs_roc[name])
    auc = roc_auc_score(all_labels_roc[name], all_probs_roc[name])
    aucs[name] = auc
    ax_roc.plot(fpr, tpr, color=disease_colors[name], linewidth=2.5,
                label=f'{name}  (AUC = {auc:.4f})')

ax_roc.set_xlabel('False Positive Rate (1 - Spesifisitas)', fontsize=12)
ax_roc.set_ylabel('True Positive Rate (Sensitivitas)', fontsize=12)
ax_roc.set_title('VisionGuard+ — ROC Curve per Penyakit\nDataset: ODIR-5K | Validation Set', fontsize=13)
ax_roc.legend(fontsize=11, loc='lower right')
ax_roc.grid(True, alpha=0.3)
ax_roc.set_xlim([0, 1])
ax_roc.set_ylim([0, 1.02])
ax_roc.text(0.55, 0.08, 'AUC > 0.80 = Good\nAUC > 0.90 = Excellent',
            fontsize=9, color='gray',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

avg_auc  = sum(aucs.values()) / len(aucs)
rows_tbl = []
for name in HEAD_NAMES:
    kat = 'Excellent' if aucs[name] >= 0.9 else 'Good' if aucs[name] >= 0.8 else 'Fair'
    rows_tbl.append([name, f'{aucs[name]:.4f}', kat])
rows_tbl.append(['RATA-RATA', f'{avg_auc:.4f}', ''])

ax_tbl.axis('off')
tbl = ax_tbl.table(cellText=rows_tbl,
                   colLabels=['Penyakit', 'AUC-ROC', 'Kategori'],
                   loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.2, 2.5)
for j in range(3):
    tbl[0, j].set_facecolor('#2C3E50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')
for j in range(3):
    tbl[len(HEAD_NAMES)+1, j].set_facecolor('#BDC3C7')
    tbl[len(HEAD_NAMES)+1, j].set_text_props(fontweight='bold')
ax_tbl.set_title('Ringkasan AUC-ROC', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('roc_curves_visionguard.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== AUC-ROC VisionGuard+ (untuk esai SEC) ===')
for name in HEAD_NAMES:
    kat = 'Excellent' if aucs[name] >= 0.9 else 'Good' if aucs[name] >= 0.8 else 'Fair'
    print(f'  {name:10}: {aucs[name]:.4f}  [{kat}]')
print(f'  {"Rata-rata":10}: {avg_auc:.4f}')
print('ROC curve disimpan: roc_curves_visionguard.png')

## Cell 12 — Risk Stratification: RUJUK / PANTAU / AMAN
Output utama VisionGuard+ untuk kader Puskesmas. Composite risk score dari probabilitas CNN asli — tanpa data sintetis.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

model.eval()
risk_records = []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs    = imgs.to(device)
        outputs = model(imgs)
        for i in range(imgs.size(0)):
            p_dr   = torch.sigmoid(outputs['DR'][i]).item()
            p_glau = torch.sigmoid(outputs['Glaukoma'][i]).item()
            p_kat  = torch.sigmoid(outputs['Katarak'][i]).item()
            p_amd  = torch.sigmoid(outputs['AMD'][i]).item()
            risk   = 0.4 * p_dr + 0.2 * p_glau + 0.2 * p_kat + 0.2 * p_amd
            risk_records.append({
                'prob_DR': p_dr, 'prob_Glaukoma': p_glau,
                'prob_Katarak': p_kat, 'prob_AMD': p_amd,
                'risk_score': risk,
                'label_DR': int(labels[i, 0].item()),
                'label_Glaukoma': int(labels[i, 1].item()),
                'label_Katarak': int(labels[i, 2].item()),
                'label_AMD': int(labels[i, 3].item()),
            })

df_risk = pd.DataFrame(risk_records)
# Threshold disesuaikan dengan distribusi skor nyata ODIR-5K
df_risk['kategori'] = pd.cut(df_risk['risk_score'],
                              bins=[0, 0.20, 0.40, 1.0],
                              labels=['AMAN', 'PANTAU', 'RUJUK'])

LABEL_ORDER = ['AMAN', 'PANTAU', 'RUJUK']
COL_RISK    = {'AMAN': '#27AE60', 'PANTAU': '#F39C12', 'RUJUK': '#E74C3C'}
PROB_COLS   = ['prob_DR', 'prob_Glaukoma', 'prob_Katarak', 'prob_AMD']
DIS_COLORS  = ['#E74C3C', '#27AE60', '#3498DB', '#9B59B6']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('VisionGuard+ Stage 1 — Risk Stratification Output\n'
             'Berbasis skor CNN EfficientNet-B4 pada ODIR-5K (Validasi: 959 pasien)',
             fontsize=14, fontweight='bold')

ax1 = axes[0][0]
counts = df_risk['kategori'].value_counts().reindex(LABEL_ORDER)
_, texts, autotexts = ax1.pie(counts.values, labels=counts.index,
                               colors=[COL_RISK[k] for k in counts.index],
                               autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight('bold')
ax1.set_title('Distribusi Pasien per Kategori Risiko', fontsize=12)

ax2 = axes[0][1]
ax2.hist(df_risk['risk_score'], bins=35, color='#3498DB', alpha=0.75, edgecolor='white', linewidth=0.5)
ax2.axvline(0.20, color=COL_RISK['PANTAU'], linestyle='--', linewidth=2.5, label='Batas PANTAU (0.20)')
ax2.axvline(0.40, color=COL_RISK['RUJUK'],  linestyle='--', linewidth=2.5, label='Batas RUJUK (0.40)')
ax2.set_xlabel('Composite Risk Score', fontsize=11)
ax2.set_ylabel('Jumlah Pasien', fontsize=11)
ax2.set_title('Distribusi Composite Risk Score', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

ax3 = axes[1][0]
mean_probs = df_risk.groupby('kategori', observed=True)[PROB_COLS].mean().reindex(LABEL_ORDER)
x = np.arange(len(LABEL_ORDER))
width = 0.2
for j, (col, dc) in enumerate(zip(PROB_COLS, DIS_COLORS)):
    ax3.bar(x + j * width, mean_probs[col], width, label=col.replace('prob_', ''), color=dc, alpha=0.85)
ax3.set_xticks(x + width * 1.5)
ax3.set_xticklabels(LABEL_ORDER, fontsize=11)
ax3.set_ylabel('Rata-rata Probabilitas CNN', fontsize=11)
ax3.set_title('Profil Penyakit per Kategori Risiko', fontsize=12)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3, axis='y')

ax4 = axes[1][1]
ax4.axis('off')
any_disease  = df_risk[['label_DR', 'label_Glaukoma', 'label_Katarak', 'label_AMD']].max(axis=1)
summary_rows = []
for kat in LABEL_ORDER:
    sub = df_risk[df_risk['kategori'] == kat]
    dr_rate  = sub['label_DR'].mean() * 100 if len(sub) > 0 else 0
    any_rate = any_disease[sub.index].mean() * 100 if len(sub) > 0 else 0
    summary_rows.append([kat, str(len(sub)), f'{len(sub)/len(df_risk)*100:.1f}%',
                         f"{sub['risk_score'].mean():.3f}", f'{dr_rate:.1f}%', f'{any_rate:.1f}%'])
tbl2 = ax4.table(cellText=summary_rows,
                 colLabels=['Kategori', 'N', 'Persen', 'Avg Risk', 'DR+', 'Ada Penyakit'],
                 loc='center', cellLoc='center')
tbl2.auto_set_font_size(False)
tbl2.set_fontsize(10)
tbl2.scale(1.1, 2.8)
for j in range(6):
    tbl2[0, j].set_facecolor('#2C3E50')
    tbl2[0, j].set_text_props(color='white', fontweight='bold')
for row_i, kat in enumerate(LABEL_ORDER):
    for j in range(6):
        tbl2[row_i + 1, j].set_facecolor(COL_RISK[kat])
        tbl2[row_i + 1, j].set_alpha(0.3)
ax4.set_title('Ringkasan Statistik per Kategori', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('risk_stratification_visionguard.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '=' * 60)
print('  VisionGuard+ — RISK STRATIFICATION SUMMARY')
print(f'  Dataset: ODIR-5K Validation ({len(df_risk)} pasien)')
print('=' * 60)
for kat in LABEL_ORDER:
    sub = df_risk[df_risk['kategori'] == kat]
    dr_rate  = sub['label_DR'].mean() * 100 if len(sub) > 0 else 0
    any_rate = any_disease[sub.index].mean() * 100 if len(sub) > 0 else 0
    print(f'\n  [{kat}] {len(sub)} pasien ({len(sub)/len(df_risk)*100:.1f}%)')
    print(f"    Avg Risk Score    : {sub['risk_score'].mean():.3f}")
    print(f'    DR Positif Nyata  : {dr_rate:.1f}%')
    print(f'    Ada Penyakit Mata : {any_rate:.1f}%')
print('\n  Tindak Lanjut VisionGuard+')
print('    AMAN   -> Skrining ulang 1 tahun, pertahankan gaya hidup sehat')
print('    PANTAU -> Kontrol 3 bulan, edukasi faktor risiko (HbA1c, tensi)')
print('    RUJUK  -> Segera rujuk ke dokter spesialis mata')
print('=' * 60)
print('Visualisasi disimpan: risk_stratification_visionguard.png')

## Cell 13 — Download Semua Hasil ke Lokal
Download semua gambar hasil dan model ke komputer kamu.

In [ ]:
from google.colab import files
import os

output_files = [
    'visionguard_model_final.pt',
    'visionguard_best.pt',
    'dataset_statistics_visionguard.png',
    'training_curve_visionguard.png',
    'metrics_table_visionguard.png',
    'roc_curves_visionguard.png',
    'gradcam_visionguard.png',
    'risk_stratification_visionguard.png',
]

for f in output_files:
    if os.path.exists(f):
        files.download(f)
        print(f'Downloaded: {f}')
    else:
        print(f'TIDAK ADA: {f} (belum dijalankan?)')